In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import time

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
from keras.losses import CategoricalCrossentropy
import csv

from loss import *
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator as DAE_OptDG
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
from plotting import *


2026-09-29 17:05:49.693230: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
import models
import importlib
importlib.reload(models)
from models import *

thresholds = [1,240,800]
# weights_path = './SQ_DAE/model-bb57b09d-SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST-checkpoints'
weights_path = '/uscms/home/jennetd/nobackup/smart-pixels/noise-paper/trained_models/SQAutoEncoder_models/model-f941a7c1-SQAutoEncoder_Transformer_40e_2bit-checkpoints'
model = CreateSQAE_MaxTransformerReg_fromweights(weights_path,
                             input_shape=(16,16,2),   
                             patch_size=(3,4),        
                             embed_dim=64,           
                             num_heads=4,            
                             ff_dim=128,              
                             num_layers=4,            
                             dropout=0.1,
                             head_dims=(256,128,64),
                             threshold_offset=0,
                             n_bits=2,
                             initial_thresholds=thresholds,
                             final_outputs=14         
                            )
model.compile(
        optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
        # loss=custom_sse_loss
        loss = custom_loss
    )


model.summary()

Loading model from /uscms/home/jennetd/nobackup/smart-pixels/noise-paper/trained_models/SQAutoEncoder_models/model-f941a7c1-SQAutoEncoder_Transformer_40e_2bit-checkpoints/weights.723-t79643.68-v85758.91.hdf5
layer_normalization_17
(None, 20, 64)
Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 raw_input (InputLayer)         [(None, 16, 16, 2)]  0           []                               
                                                                                                  
 soft_quantizer_output (SoftQua  (None, 16, 16, 2)   4           ['raw_input[0][0]']              
 ntizeLayer)                                                                                      
                                                                                                  
 patch_extractor_1 (PatchExtrac  (None, None

In [11]:
import importlib
import DAE_DataGenerator
import dataset_utils
import SoftQuantizeLayer
import AnnealingScheduler

importlib.reload(DAE_DataGenerator)
importlib.reload(dataset_utils)
importlib.reload(SoftQuantizeLayer)
importlib.reload(AnnealingScheduler)

from AnnealingScheduler import AnnealingScheduler
# from DAE_DataGenerator import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
#from dataprep import *
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *

In [12]:
dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "tfrecords")

# dataset_dir_train = os.path.join(dataset_base_dir, "/dataset_3sr_16x16_50x12P5_parquets", 'train/')
# dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets", 'test/')

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

tfrecords_dir_train   = os.path.join(tfrecords_base_dir, "TFR_train",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')


tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3srCI_16x16_40eN_raw')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3srCI_16x16_40eN_raw')



print(dataset_dir_val)
batch_size = 5000
val_batch_size = 5000
train_file_size = 80
val_file_size = 20

#### Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=False,
    # seed=42,
    quantize=False,
    # bitwise_categorical = True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=False,
    # seed=42,
    quantize=False,
    # bitwise_categorical = True,
)

/uscms/home/bweiss/nobackup/smart-pixels/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/test_contained/
Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_val/3srCI_16x16_40eN_raw/metadata.json


Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_train/3srCI_16x16_40eN_raw/metadata.json


In [13]:
from datetime import datetime
tname = 'SQDAE_SlimTransformer_Regression_2bit_40eN_Target_0_30_845_JENNET_TEST'#0_7_23_134_404_867_1748'
fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./SQ_DAE_regression//model-{fingerprint}-{tname}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.4f}-v{val_loss:.4f}.hdf5'
print("fingerprint: ", fingerprint)

fingerprint:  ca4e77a0


In [14]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

early_stopping_patience = 1000

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort(key=lambda x: os.path.getmtime(os.path.join(base_dir,x)))
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=400.0, 
    verbose=1      
)

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)


In [15]:
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=5000,
        shuffle=False,
        verbose=1
    )

Epoch 1/5000
 2/84 [..............................] - ETA: 7s - loss: 97956.3203   

2026-09-29 17:10:47.795767: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x55c92a733560


41/84 [=============>................] - ETA: 4s - loss: 17733.8535

KeyboardInterrupt: 